# Reliable AI System Evaluation with ASI

**ASI (Active Statistical Inference)** combines a small set of expensive true evaluation labels with a large pool of cheap proxy evaluation labels to produce a statistically valid, bias-corrected quality metric. It goes one step further than basic bias correction: when your LLM judge reports not only a label but also an **uncertainty score**, ASI uses that score to concentrate your annotation budget on the conversations where the judge is *least reliable* — extracting more information from the same number of human annotations.

---

**What you will learn:**

- Why LLM-as-Judge metrics are systematically biased
- How to exploit judge uncertainty scores to allocate your annotation budget efficiently
- How to use ASI to produce a valid, bias-corrected metric with a tight confidence interval

## The problem: your LLM judge disagrees with your users

Let's assume you run a customer-facing agentic assistant handling thousands of conversations per day.

### The signals

- **Every tenth user** reports incorrect or fabricated information.
- You deploy an LLM judge to measure the hallucination rate. It tells you the rate is **5%**.

The users and the judge **disagree**, the latter is *systematically optimistic*. 

### Fixing an annotation budget

Manual annotation is expensive. With thousands of conversations per day, annotating them all is not an option. You fix a budget of **200 manual annotations** — accurate ground truth, but scarce. The obvious approach is to pick 200 conversations at random, yet a smarter strategy may exist.

### The judge's hidden signal

Your LLM judge is more sophisticated than a plain yes/no classifier. For each conversation it evaluates, it also outputs an **uncertainty score**: a measure of how confident it is in its own label. Some conversations are clear-cut; others are genuinely ambiguous, and the judge knows it.

This uncertainty signal is valuable: it tells you *where* a human annotator's time would be best spent.

### The challenge

You now have:
- **2,200 LLM judgements** — cheap and fast, but biased
- **Budget for 200 human annotations** — to be spent wisely on accurate and informative instances.

The question is not just *how many* to annotate, but **which ones**.

- **Uniform sampling** — pick 200 conversations at random. Simple, but wastes budget on conversations the judge already handles reliably.
- **Uncertainty-guided (active) sampling** — pick conversations where the judge is *most uncertain*. Your annotation budget goes where it adds the most information.

**ASI** formalises the second strategy with a statistically valid correction for both the judge's bias and the non-uniform selection, yielding an unbiased estimate with a tighter confidence interval.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from glide.core.dataset import Dataset
from glide.core.simulated_datasets import generate_binary_dataset_with_oracle_sampling
from glide.estimators import ASIMeanEstimator, ClassicalMeanEstimator
from glide.samplers import ActiveSampler

# ── Colour palette ──────────────────────────────────────────
C_JUDGE = "#E74C3C"  # LLM judge        — red-orange
C_HUMAN = "#2E86AB"  # Human-only       — blue
C_GLIDE = "#27AE60"  # GLIDE / ASI      — green
C_TRUTH = "#2C3E50"  # True value       — dark slate
C_ACTIVE = "#E67E22"  # Active sampling  — amber

# ── Global plot style ───────────────────────────────────────
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAFAFA",
        "axes.grid": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.8,
        "axes.titlesize": 14,
        "axes.titlepad": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
    }
)

## Simulating 2,200 conversations with a biased judge that reports uncertainty

`generate_binary_dataset_with_oracle_sampling` produces a synthetic dataset that mirrors the scenario above.

Each **record** represents one conversation and carries three fields:

| Field | Meaning |
|-------|---------|
| `y_true` | Ground-truth binary label: $Y_i = 1$ = hallucination confirmed by a human annotator |
| `y_proxy` | Binary label from the LLM judge: $\tilde{Y}_i = 1$ = hallucination flagged |
| `uncertainty` | Oracle uncertainty score: $\sqrt{\mathbb{E}[(\tilde{Y}_i - Y_i)^2 \mid x_i]}$ |

where $Y_i$ is the human annotation, $\tilde{Y}_i$ is the LLM judge annotation, and $x_i$ is the conversation in question. High `uncertainty` means the judge is *least reliable* for that conversation — exactly where a human annotator adds the most value.

> **Simulated annotation:** In practice, `y_true` is revealed only after a human annotates the conversation. Here we generate the ground-truth labels upfront so that we can later simulate the annotation process by selectively revealing them.

In [ ]:
N_TOTAL = 2200  # total conversations evaluated by the LLM judge
N_LABELED = 200  # human annotation budget
TRUE_RATE = 0.10  # true hallucination rate (unknown in practice)
RANDOM_SEED = 0

oracle_pool = generate_binary_dataset_with_oracle_sampling(
    N=N_TOTAL,
    true_mean=0.10,
    proxy_mean=0.05,
    correlation=0.55,
    random_seed=RANDOM_SEED,
)

In [ ]:
print(f"Total conversations  : {len(oracle_pool):,}")
print(f"Annotation budget    : {N_LABELED}")
print()
print("Sample records")
for record in oracle_pool[:3]:
    print(f"  {record}")

## The judge's uncertainty varies widely across conversations

Before deciding *which* conversations to annotate, let's inspect the distribution of the judge's uncertainty scores. A uniform distribution would mean every conversation is equally hard for the judge; the actual distribution reveals that some conversations are genuinely ambiguous.


In [ ]:
uncertainties = np.array(oracle_pool["uncertainty"])
pi_uniform = N_LABELED / N_TOTAL

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.hist(uncertainties, bins=40, color=C_ACTIVE, alpha=0.75, label="Judge uncertainty")
ax.axvline(
    uncertainties.mean(),
    color=C_TRUTH,
    linestyle="--",
    linewidth=2,
    label=f"Mean uncertainty = {uncertainties.mean():.3f}",
)

ax.set_xlabel("Judge uncertainty score")
ax.set_ylabel("Number of conversations")
ax.set_title("Distribution of LLM Judge Uncertainty Across 2,200 Conversations")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(
    f"Uncertainty  — min: {uncertainties.min():.3f}  |  "
    "mean: {uncertainties.mean():.3f}  |  max: {uncertainties.max():.3f}"
)

## Smart choice of conversations to annotate with `ActiveSampler`

`ActiveSampler` translates the uncertainty scores into **Bernoulli sampling probabilities**:

$$\pi_i = \min\!\left(1,\; \frac{\text{budget} \times \text{uncertainty}_i}{\sum_j \text{uncertainty}_j}\right)$$

Each record is then independently selected with probability $\pi_i$. The expected number of selected records equals the budget.  
Records with high uncertainty receive the highest $\pi_i$ and are therefore most likely to be annotated.

The sampler returns the full dataset enriched with two new fields:

| Field | Meaning |
|-------|---------|
| `pi`  | Sampling probability $\pi_i \in (0, 1]$ for this record |
| `xi`  | Bernoulli draw: `1` if the record was selected for annotation, `0` otherwise |

In [ ]:
# ── Step 1: compute sampling probabilities and draw annotation indicators ──
sampled_pool = ActiveSampler().sample(
    oracle_pool,
    uncertainty_field="uncertainty",
    budget=N_LABELED,
    random_seed=RANDOM_SEED,
)

# ── Step 2: simulate annotation — reveal y_true only for selected records ──
asi_records = []
for record in sampled_pool.records:
    asi_record = {"y_proxy": record["y_proxy"], "pi": record["pi"]}
    if record["xi"] == 1:
        asi_record["y_true"] = record["y_true"]  # human annotation revealed
    asi_records.append(asi_record)

asi_dataset = Dataset(asi_records)

n_annotated = sum(1 for r in asi_dataset.records if "y_true" in r)
pi_values = np.array(sampled_pool["pi"])

In [ ]:
print(f"Total conversations          : {N_TOTAL:,}")
print(f"Annotation budget (expected) : {N_LABELED}")
print(f"Actually annotated (realized): {n_annotated}")
print()
print(
    f"Sampling probabilities  — min: {pi_values.min():.3f}  |  "
    f"mean: {pi_values.mean():.3f}  |  max: {pi_values.max():.3f}"
)
print(f"Uniform pi (for reference)   : {pi_uniform:.3f}")

## Two naive strategies both fail

Before deploying ASI, consider two simpler approaches:

**Option A — Trust the judge on all conversations.**  
Precise (large sample), but the judge's systematic bias makes the estimate wrong.

**Option B — Trust only the 200 uniformly sampled human annotations.**  
Unbiased, but the 95% confidence interval is wide.

In [ ]:
# Option A: LLM judge — average proxy labels over all 2,200 conversations
judge_estimate = ClassicalMeanEstimator().estimate(dataset=oracle_pool, y_field="y_proxy")
judge_mean = judge_estimate.mean
judge_lower = judge_estimate.confidence_interval.lower_bound
judge_upper = judge_estimate.confidence_interval.upper_bound

# Option B: human labels on a uniform random sample of 200 conversations
rng_uniform = np.random.default_rng(RANDOM_SEED)
uniform_idx = rng_uniform.choice(N_TOTAL, size=N_LABELED, replace=False)
uniform_labeled_ds = Dataset([oracle_pool.records[i] for i in uniform_idx])
human_estimate = ClassicalMeanEstimator().estimate(dataset=uniform_labeled_ds, y_field="y_true")
human_mean = human_estimate.mean
human_lower = human_estimate.confidence_interval.lower_bound
human_upper = human_estimate.confidence_interval.upper_bound

In [ ]:
sep = "-" * 66
print(sep)
print(f"{'Method':<36} {'Estimate':>8}   {'95% confidence interval':>16}")
print(sep)
print(f"{'LLM Judge only  (N=2,200)':<36} {judge_mean:>7.1%}   [{judge_lower:.1%}, {judge_upper:.1%}]")
print(f"{'Human labels only (n=200, uniform)':<36} {human_mean:>7.1%}   [{human_lower:.1%}, {human_upper:.1%}]")
print(sep)
print(f"{'True rate  (simulation)':<36} {'10.0%':>8}")

## ASI corrects annotation bias and exploits uncertainty for tighter intervals

`ASIMeanEstimator` implements Active Statistical Inference, which:

1. **Measures and removes the judge's systematic bias** — on the annotated subset, it computes the gap between proxy and true labels and uses it to correct the estimate across all conversations.
2. **Corrects for non-uniform sampling** — using Inverse Probability Weighting (IPW): each labeled record's contribution is scaled by $1/\pi_i$, so records that were *less likely* to be selected count more in the estimate, restoring statistical validity.

The combined effect is an estimator that is both **unbiased** and **more efficient** than uniform human-only annotation.

In [ ]:
estimator = ASIMeanEstimator()

asi_result = estimator.estimate(
    asi_dataset,
    y_true_field="y_true",
    y_proxy_field="y_proxy",
    sampling_probability_field="pi",
    metric_name="Hallucination Rate",
    confidence_level=0.95,
)

In [ ]:
print(asi_result.summary())

## ASI Delivers an Unbiased Estimate at Low Cost

The plot below compares **point estimates** and **95% confidence intervals** for all three methods against the true hallucination rate (dashed line):

- **LLM judge**: very narrow confidence interval, but wrong.
- **Human-only (uniform)**: unbiased, but the confidence interval is wide.
- **ASI**: unbiased *and* narrower — the bias correction from the proxy labels combined with smarter, uncertainty-guided annotation.

In [ ]:
estimates = [
    (
        f"LLM Judge\n(N={N_TOTAL:,}  |  raw proxy)",
        judge_mean,
        judge_lower,
        judge_upper,
        C_JUDGE,
    ),
    (
        f"Human Annotation\n(n={N_LABELED} |  uniform sample)",
        human_mean,
        human_lower,
        human_upper,
        C_HUMAN,
    ),
    (
        f"ASI (GLIDE)\n(n={n_annotated}  active  +  N={N_TOTAL:,})\n(uncertainty-guided)",
        asi_result.mean,
        asi_result.confidence_interval.lower_bound,
        asi_result.confidence_interval.upper_bound,
        C_GLIDE,
    ),
]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
y_pos = [2, 1, 0]

for y, (label, mean, lo, hi, color) in zip(y_pos, estimates):
    ax.plot([lo, hi], [y, y], color=color, linewidth=4, solid_capstyle="round", zorder=3)
    for xc in [lo, hi]:
        ax.plot([xc, xc], [y - 0.2, y + 0.2], color=color, linewidth=2.5, zorder=3)
    ax.scatter(mean, y, s=200, color=color, zorder=5, edgecolors="white", linewidths=2.5)
    ax.text(mean, y + 0.34, f"{mean:.1%}", ha="center", va="bottom", fontsize=12, color=color, fontweight="bold")
    ax.text(mean, y - 0.34, f"[{lo:.1%},  {hi:.1%}]", ha="center", va="top", fontsize=11, color="#888888")

ax.axvline(TRUE_RATE, color=C_TRUTH, linestyle="--", linewidth=2.5, zorder=4)
ax.text(TRUE_RATE + 0.004, 2.72, "True rate  10%", color=C_TRUTH, fontsize=10.5, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels([e[0] for e in estimates], fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Estimated Hallucination Rate", fontsize=12)
ax.set_title("GLIDE ASI Delivers an Unbiased, Precise Estimate", fontsize=14, fontweight="bold")
ax.set_xlim(-0.01, 0.26)
ax.set_ylim(-0.8, 3.2)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

## Testing whether the hallucination rate is within acceptable limits

GLIDE's estimation output enables a formal hypothesis test: is the true hallucination rate significantly higher than 5%?

$$H_0 : \mu = 5\% \qquad H_1 : \mu > 5\%$$

Without ASI, we can use the LLM judge estimate only — which gives a misleading confidence interval close to 5%, making it unlikely that we would reject the null hypothesis. The human-only confidence interval is wide and leads to conservative decisions. ASI combines both sources using a statistically valid method that enables accurate hypothesis testing.

In [ ]:
z_stat, p_value, _ = asi_result.confidence_interval.test_null_hypothesis(
    h0_value=0.05,  # LLM judge's claimed rate
    alternative="larger",  # H1: true rate > 5%
)

In [ ]:
sep = "-" * 48
print("Hypothesis test — ASI Estimator")
print(sep)
print("H0 : hallucination rate = 5%   (LLM says so)")
print("H1 : hallucination rate > 5%   (users complain!)")
print()
print(f"z-statistic : {z_stat:.2f}")
print(f"p-value     : {p_value:.10f}")
print()
if p_value < 0.05:
    print("Decision  : Given the p-value is lower than our threshold (p-value < 0.05), we reject H0 at the 5% level.")
    print("=> The true hallucination rate is significantly above 5%.")
else:
    print(
        "Decision  : Given the p-value is higher than our threshold (p-value >= 0.05), "
        "we cannot reject H0 at the 5% level."
    )

Notice that the null hypothesis is rejected, signalling that the hallucination rate is significantly above the fixed threshold.

Let us try the same hypothesis test using human annotations only.

In [ ]:
z_stat_cm, p_value_cm, _ = human_estimate.confidence_interval.test_null_hypothesis(
    h0_value=0.05,
    alternative="larger",
)

In [ ]:
sep = "-" * 48
print("Hypothesis test — Classical Mean Estimator (Human labels only, uniform sample)")
print(sep)
print("H0 : hallucination rate = 5%   (LLM says so)")
print("H1 : hallucination rate > 5%   (users complain!)")
print()
print(f"z-statistic : {z_stat_cm:.2f}")
print(f"p-value     : {p_value_cm:.10f}")
print()
if p_value_cm < 0.05:
    print("Decision  : Given the p-value is lower than our threshold (p-value < 0.05), we reject H0 at the 5% level.")
    print("=> The true hallucination rate is significantly above 5%.")
else:
    print(
        "Decision  : Given the p-value is higher than our threshold (p-value >= 0.05), "
        "we cannot reject H0 at the 5% level."
    )

The null hypothesis may not be rejected due to high uncertainty — with only 200 uniformly chosen annotations, the confidence interval is too wide to draw a firm conclusion. ASI fixes this in two complementary ways: it uses the LLM judge labels to reduce uncertainty, *and* it concentrates the annotation budget on the most informative conversations.

## Summary: ASI combines accuracy, efficiency, and smart sampling

| | LLM Judge | Human-only (uniform) | **ASI** |
|-|-----------|---------------------|----------|
| Sample size | 2,200 LLM-judge annotations | 200 Human annotations | 2,200 LLM-judge + ~200 Human annotations|
| Unbiased estimate | ❌ | ✅ | ✅ |
| Narrow confidence interval | 🟠 *(misleading)* | ❌ | ✅ |
| Uses uncertainty signal | ❌ | ❌ | ✅ |
| Labeling cost | Low | High and inefficient | High but optimized for efficiency |

**Key takeaways:**

1. **LLM judges are biased.** A narrow confidence interval around the wrong value is worse than useless — it gives false confidence.

2. **The uncertainty score is an asset.** When your judge reports how confident it is, `ActiveSampler` turns that signal into a smarter annotation strategy: the same budget covers more ground by focusing on the hardest conversations.

3. **ASI is valid under non-uniform sampling.** Inverse Probability Weighting corrects for the fact that uncertain records are over-represented among the annotated set, preserving statistical validity.

4. **200 active annotations outperform 200 uniform annotations.** The efficiency gain grows with the quality and variability of the judge's uncertainty scores.

---

*Want to go further? The [ASI scientific validation notebook](../deep_dive/asi_validation.ipynb) provides rigorous empirical evidence — coverage validity and confidence interval width across a sweep of proxy correlations and confidence levels.*